In [ ]:
import sys, os
# Imports of vbn_utils, ccf_utils, decoding_utils, analysis_utils,
# notebook_utils, vbn_4day_utils resolve via vbn_code/utilities/.
_UTILS_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'utilities'))
if _UTILS_DIR not in sys.path:
    sys.path.insert(0, _UTILS_DIR)

import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import nrrd
import scipy.stats
from scipy.stats import binned_statistic_2d, binned_statistic
import decoding_utils as du
import vbn_utils as vbn
import vbn_utils
import h5py
import ccf_utils
from matplotlib.colors import LinearSegmentedColormap, to_rgba
%matplotlib inline

In [ ]:
from notebook_utils import scatter_ccf, binned_stat_ccf, weighted_gaussian_filter

In [ ]:
high_res = True
if high_res:
    plt.rcParams['figure.dpi'] = 300
    plt.rcParams['savefig.dpi'] = 300
    plt.rcParams['font.size'] = 12
    plt.rcParams['pdf.fonttype'] = 42
    
    plt.rcParams['figure.facecolor'] = 'white'
    plt.rcParams['axes.facecolor'] = 'white'
    plt.rcParams['savefig.facecolor'] = 'white'  # affects clipboard copy too
    plt.rcParams['savefig.transparent'] = False

## Data loading

In [ ]:
#Paths to all of the useful supplemental tables and tensors
active_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor.hdf5"
passive_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor_passive.hdf5"
opto_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbn_opto_tensor_unit_grouped.hdf5"

stim_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
unit_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_units_with_responsiveness.csv"
unit_table_with_rf_stats = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/units_with_rf_stats.csv"
unit_table_opto_metrics = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/unit_opto_metrics.csv"
unit_cluster_labels = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/unit_cluster_labels.csv"
unit_vis_responsive = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/units_with_vis_responsiveness.csv"

sessions_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_sessions_table.csv"

In [ ]:
units = pd.read_csv(unit_table_file)

stim_table = pd.read_csv(stim_table_file)
stim_table = stim_table.drop(columns='Unnamed: 0')

sessions_table = pd.read_csv(sessions_table_file)

## Load GLM dropout scores

In [ ]:
# dropouts = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/GLM_dropout_with_unit_info.csv")
dropouts = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/GLM_dropout_with_unit_info_active_passive.csv")

## CCF coordinate setup

In [ ]:
import nrrd
import ccf_utils as ccf
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import to_rgba

annotation_volume = nrrd.read("/Volumes/programs/mindscope/workgroups/np-behavior/annotation_25.nrrd")[0]
structure_tree = pd.read_csv("/Volumes/programs/mindscope/workgroups/dynamicrouting/dynamic_gating_insertions/ccf_structure_tree_2017.csv")

def get_area_mask(area, annotation_volume, structure_tree):

    if len(structure_tree[structure_tree['acronym']==area]) == 0:
        print(f'could not find area {area} in structure tree')
        return
    
    area_id = structure_tree[structure_tree['acronym']==area]['id'].values[0]
    if not area_id in annotation_volume:
        area_ccf = structure_tree[structure_tree['parent_structure_id']==area_id]
        area_mask = np.sum([annotation_volume==id for id in area_ccf['id'].values], axis=0)
    else:
        area_mask = annotation_volume==area_id
    
    return area_mask

def get_area_projection(area, annotation_volume, structure_tree, plane, spacing=25, padding=0):

    axis_to_average = {'coronal': 0, 'sagittal': 2, 'horizontal': 1}

    if isinstance(area, list):
        area_mask = np.full_like(annotation_volume, False)
        for a in area:
            amask = get_area_mask(a, annotation_volume, structure_tree)
            area_mask = np.logical_or(area_mask, amask)
        #area_mask = np.sum([get_area_mask(a, annotation_volume, structure_tree) for a in area], axis=0)
    else:
        area_mask = get_area_mask(area, annotation_volume, structure_tree)
    
    area_projection = np.nanmax(area_mask, axis=axis_to_average[plane])
    if plane in ['coronal', 'horizontal']:
        area_projection = area_projection[:, :228]
    
    rows, cols = np.where(area_projection)
    min_row, max_row = np.min(rows)-padding, np.max(rows)+padding
    min_col, max_col = np.min(cols)-padding, np.max(cols)+padding
    bounding_box = np.array([min_row, max_row, min_col, max_col])

    scale_factor = 25//spacing
    projection_box = np.repeat(np.repeat(area_projection[min_row:max_row+1, min_col:max_col+1], scale_factor, axis=0), scale_factor, axis=1)
    if plane in ['coronal', 'horizontal']:
        projection_box = np.flipud(projection_box)
        bounding_box = bounding_box[[2, 3, 0, 1]]
    else:
        projection_box = np.flipud(projection_box.T)
        
    return projection_box, bounding_box*scale_factor


def get_border(array, coord):

    if len(coord)==3:
        x, y, z = coord
        array_box = np.logical_not(array[x-1:x+2, y-1:y+2, z-1:z+2])
    elif len(coord)==2:
        x, y = coord
        array_box = np.logical_not(array[x-1:x+2, y-1:y+2])

    return any(array_box.flatten())


def get_area_boundary(area, annotation_volume, structure_tree, plane, spacing=25):
    area_projection, bounding_box = get_area_projection(area, annotation_volume, structure_tree, plane, spacing, padding=2)
    
    bounding_box = np.array(bounding_box)*25

    coords = np.where(area_projection)
    area_edges = np.zeros(area_projection.shape)
    for coord in tuple(zip(*coords)):
        area_edges[coord] = get_border(area_projection, coord)

    return area_edges, bounding_box


def draw_area_boundary(area, plane, area_edges, bounding_box, ax=None, cmap=None):

    ax_lims = None
    if ax is None:
        fig, ax = plt.subplots()
    else:
        ax_lims = ax.get_xlim(), ax.get_ylim()

    if cmap is None:
        color = ccf.get_area_color(vbn.make_iterable(area)[0], structure_tree)
        colors = [(1, 1, 1, 0), to_rgba(color)]  # RGBA for transparent (0) and red (1)
        n_bins = 10  # Discretize the interpolation into bins
        area_cmap = LinearSegmentedColormap.from_list('area_map', colors, N=n_bins)
    else:
        area_cmap = cmap
    
    if plane == 'coronal':
        
        im = ax.imshow(area_edges, extent=[bounding_box[0], bounding_box[1], bounding_box[2], bounding_box[3]], 
                       cmap=area_cmap, aspect='auto')
        #ax.invert_xaxis()
        ax.set_xlabel('left->right')
        ax.set_ylabel('ventral->dorsal')

    elif plane == 'sagittal':
        im = ax.imshow(area_edges, extent=[bounding_box[0], bounding_box[1], bounding_box[2], bounding_box[3]], 
                       cmap=area_cmap, aspect='auto')
        ax.set_xlabel('anterior->posterior')
        ax.set_ylabel('ventral->dorsal')
    
    elif plane == 'horizontal':
        im = ax.imshow(area_edges, extent=[bounding_box[0], bounding_box[1], bounding_box[2], bounding_box[3]], 
                       cmap=area_cmap, aspect='auto')
        #ax.invert_xaxis()
        ax.set_xlabel('left->right')
        ax.set_ylabel('posterior->anterior')

    if ax_lims:
        ax.set_xlim(ax_lims[0])
        ax.set_ylim(ax_lims[1])
    
    return ax
        

## Annotate dropout data

In [ ]:
dropouts['has_ccf'] = dropouts['left_right_ccf_coordinate']>0
dropouts['cortical_layer'] = dropouts['cortical_layer'].replace('3-Feb','2/3') # necessary since 2/3 sometimes gets incorrectly reformatted as a date
dropouts['cortical_layer'] = dropouts['cortical_layer'].replace('1','2/3')
dropouts['cortical_layer'] = dropouts['cortical_layer'].replace('6a','6')
dropouts['cortical_layer'] = dropouts['cortical_layer'].replace('6b','6')


## CCF spatial distribution of lick vs. image dropout scores

In [ ]:
plt.rcParams['font.size'] = 18

In [ ]:
def plot_area_ccf_cluster_distribution(area, metric, plane='sagittal', binsize=100, clim='auto'):
    fig, ax = plt.subplots(2,2, figsize=(10,10))
    fig.suptitle(f'{area}, {metric}')
    projection = get_area_projection(area, annotation_volume, structure_tree, plane=plane, spacing=25)
    ax[0][0].imshow(projection[0], extent=np.array((projection[1][0], projection[1][1], projection[1][2], projection[1][3]))*25, aspect='auto', cmap='gray', alpha=0.1)    
    
    area_neurons = vbn.get_unit_ids(dropouts[(dropouts['has_ccf'])&(dropouts['firing_rate']>=0.5)], area)
    area_dropouts = dropouts[dropouts['unit_id'].isin(area_neurons)]
    print(f'{area} mean: {area_dropouts[metric].mean()}')
    if clim=='auto':
        scatter_ccf(area_dropouts, plane=plane, c=metric, size=15, ax=ax[0][0],)# clim=[0,-0.05] )
    else:
        scatter_ccf(area_dropouts, plane=plane, c=metric, size=15, ax=ax[0][0], clim=clim)
    
    ax[0][1].imshow(projection[0], extent=np.array((projection[1][0], projection[1][1], projection[1][2], projection[1][3]))*25, aspect='auto', cmap='gray', alpha=0.1)    
    values, _, _ = binned_stat_ccf(area_dropouts, binsize=binsize, plane=plane, c=metric, statistic=np.nanmean, ax=ax[0][1],)
    counts, x_edge, y_edge = binned_stat_ccf(area_dropouts, binsize=binsize, plane=plane, c=metric, statistic='count', ax=ax[1][0])
    
    ax[1][1].imshow(projection[0], extent=np.array((projection[1][0], projection[1][1], projection[1][2], projection[1][3]))*25, aspect='auto', cmap='gray', alpha=0.1)    
    smoothed = weighted_gaussian_filter(values, counts, 1).T
    im = ax[1][1].imshow(-smoothed, extent=np.array((x_edge[0], x_edge[-1], y_edge[-1], y_edge[0])), aspect='auto', cmap='viridis_r',)
    for area in vbn.make_iterable(area):
        area_boundary, bounding_box = get_area_boundary(area, annotation_volume, structure_tree, plane, spacing=25)
        for a in ax.flatten():
            draw_area_boundary(area, plane, area_boundary, bounding_box, ax=a)
            a.set_aspect('equal')

    divider = make_axes_locatable(ax[1][1])
    cbar_ax = divider.append_axes("right", size="5%", pad=0.1)  # Adjust size and padding
    colorbar = fig.colorbar(im, cax=cbar_ax)
    # plt.colorbar(im, ax=ax[1][1])

    ax[0][0].set_title('individual units')
    ax[0][1].set_title('binned mean')
    ax[1][0].set_title('binned count')
    ax[1][1].set_title('binned mean smoothed')

    plt.tight_layout()


In [ ]:
plt.rcParams['font.size'] = 18
plot_area_ccf_cluster_distribution(['SCm', 'MRN'], 
                                   "('absolute_change_from_full', 'licks')", 
                                   plane='sagittal', binsize=100)

In [ ]:
plot_area_ccf_cluster_distribution(['SCm', 'MRN'], 
                                   "('absolute_change_from_full', 'all-images')", 
                                   plane='sagittal', binsize=100)

In [ ]:
plot_area_ccf_cluster_distribution(['VISam', 'VISpm'], 
                                   "('absolute_change_from_full', 'licks')", 
                                   plane='sagittal', binsize=50 ,clim=[0, 0.05])

In [ ]:
fig, ax = plt.subplots()
for ia, area in enumerate(['VISp', 'VISl', 'VISal', 'VISpm', 'VISam', 'VISrl']):
    lick_vals = dropouts[dropouts['structure_acronym']==area]["('absolute_change_from_full', 'licks')"].values
    
    iteration_proportions = []
    for iteration in range(1000):
        iter_vals = np.random.choice(lick_vals, size=len(lick_vals), replace=True)
        proportion_high_lick_units = np.sum(iter_vals<-0.01)/len(iter_vals)
        iteration_proportions.append(proportion_high_lick_units)
    
    
    ax.boxplot(iteration_proportions, positions=[ia], widths=0.5, patch_artist=True, 
               boxprops=dict(facecolor='gray'), showfliers=False, whis=[10, 90], medianprops=dict(color='black'), notch=True)

ax.set_xticks(np.arange(len(['VISp', 'VISl', 'VISal', 'VISpm', 'VISam', 'VISrl'])))
ax.set_xticklabels(['VISp', 'VISl', 'VISal', 'VISpm', 'VISam', 'VISrl'])
ax.set_ylabel('Proportion of units \n with lick dropout > 0.01')
vbn.formatFigure(fig, ax)

## Overall GLM model performance across regions

In [ ]:
quality = du.apply_unit_quality_filter(units, no_abnorm=True)
quality_units = units[quality]

quality_units = quality_units[quality_units['brain_division'] != 'not in list']
quality_units.loc[quality_units['structure_acronym'].isin(['SCig', 'SCiw']).astype(bool), 'structure_acronym'] = 'SCm'
quality_units.loc[quality_units['structure_acronym'].isin(['MGv', 'MGd', 'MGm']).astype(bool), 'structure_acronym'] = 'MG'


In [ ]:
areas_meet_threshold = quality_units['structure_acronym'].value_counts() > 100
areas_meet_threshold = areas_meet_threshold[areas_meet_threshold].index
areas_meet_threshold

## GLM variance explained by area

In [ ]:
dropouts_areas_100 = dropouts[dropouts['structure_acronym'].isin(areas_meet_threshold)]
sorted_areas = dropouts_areas_100.groupby('structure_acronym').median()["('variance_explained', 'Full')"].sort_values(ascending=True).index.values

plt.rcParams['font.size'] = 16
fig, ax = plt.subplots(figsize=(12, 6))
for ia, area in enumerate(sorted_areas):
    area_data = dropouts_areas_100[dropouts_areas_100['structure_acronym'] == area]
    area_color = ccf.get_area_color(area, structure_tree)
    ax.boxplot(area_data["('variance_explained_full', 'Full')"].values, positions=[ia], widths=0.5, patch_artist=True, boxprops=dict(facecolor=area_color), medianprops=dict(color='k', linewidth=1),
                showfliers=False, whis=[10, 90], notch=True)
vbn.formatFigure(fig, ax, yLabel='Variance Explained (cross-validated)')
ax.set_xticks(np.arange(len(sorted_areas)))
ax.set_xticklabels(sorted_areas, rotation=90)

In [ ]:
dropouts_areas_edited = dropouts.merge(quality_units[['unit_id', 'structure_acronym']], left_on='unit_id', right_on='unit_id', suffixes=('', '_edited'))

In [ ]:
areas_meet_threshold = quality_units['structure_acronym'].value_counts() > 1000
areas_meet_threshold = areas_meet_threshold[areas_meet_threshold].index
areas_meet_threshold_units = vbn.get_unit_ids(quality_units, areas_meet_threshold)

In [ ]:
fig, axes = plt.subplots(2,1)
ax = axes[0]
ax2 = axes[1]
fig.set_size_inches(12,5)
area_dropout_pivot = dropouts_areas_edited[dropouts_areas_edited['structure_acronym_edited'].isin(areas_meet_threshold)].pivot_table(index='structure_acronym_edited', 
                                    values=["('absolute_change_from_full', 'all-images')","('absolute_change_from_full', 'licks')", '(\'variance_explained\', \'Full\')'], aggfunc='mean')

#area_dropout_pivot = dropouts[dropouts['firing_rate']>0.5].pivot_table(index='structure_acronym', values=["('absolute_change_from_full', 'all-images')","('absolute_change_from_full', 'licks')"], aggfunc='mean')
areas_ordered = area_dropout_pivot.sort_values(by="('absolute_change_from_full', 'all-images')").index.values

for ia, area in enumerate(areas_ordered):
    vis_vals = -dropouts_areas_edited[dropouts_areas_edited['structure_acronym_edited']==area]["('absolute_change_from_full', 'all-images')"].values
    lick_vals = -dropouts_areas_edited[dropouts_areas_edited['structure_acronym_edited']==area]["('absolute_change_from_full', 'licks')"].values

    area_color = ccf_utils.get_area_color(area, structure_tree)
    ax.boxplot(vis_vals, positions=[ia], widths=0.5, patch_artist=True, showfliers=False, 
               whis=[10, 90], boxprops=dict(facecolor=area_color), medianprops=dict(color='k', linewidth=1), notch=True)
    ax2.boxplot(lick_vals, positions=[ia], widths=0.5, patch_artist=True, showfliers=False, 
                whis=[10, 90],  boxprops=dict(facecolor=area_color), medianprops=dict(color='k', linewidth=1), notch=True)

ax.set_title('Image dropout scores')
ax2.set_title('Lick dropout scores')

ax.xaxis.set_visible(False)
    # vp1 = ax.violinplot(vis_vals, positions=[ia], widths=0.5,
    #               )
    # vp2 = ax2.violinplot(lick_vals, positions=[ia], widths=0.5,)

    # for vp in [vp1, vp2]:
    #     for pc in vp['bodies']:
    #         pc.set_facecolor(area_color)
    #     for pc, pcitem in vp.items():
    #         if isinstance(pcitem, list):
    #             [item.set_color(area_color) for item in pcitem]
    #         else:
    #             pcitem.set_edgecolor(area_color)
    
ax2.set_xticks(np.arange(len(areas_ordered)))
ax2.set_xticklabels(areas_ordered, rotation=90)

[vbn.formatFigure(fig, a,) for a in [ax, ax2]]
# area_dropout_pivot.sort_values(by="('absolute_change_from_full', 'all-images')")["('absolute_change_from_full', 'all-images')"].plot(kind='bar', ax=ax)
# area_dropout_pivot.sort_values(by="('absolute_change_from_full', 'all-images')")["('absolute_change_from_full', 'licks')"].plot(kind='bar', ax=ax2, color='r', alpha=0.5)
# ax.invert_yaxis()

# fig, ax = plt.subplots()
# area_dropout_pivot.sort_values(by='(\'variance_explained\', \'Full\')')['(\'variance_explained\', \'Full\')'].plot(kind='bar', ax=ax)

# ax2.set_ylim(0, -0.005)
fig, ax = plt.subplots()
for ia, area in enumerate(areas_ordered):
    if not 'VIS' in area:
        continue
    lick_vals = dropouts_areas_edited[dropouts_areas_edited['structure_acronym_edited']==area]["('absolute_change_from_full', 'licks')"].values
    print(f'{area} {np.sum(lick_vals<-0.01)/len(lick_vals)} {np.median(lick_vals)}')
    area_color = ccf_utils.get_area_color(area, structure_tree)
    ax.boxplot(lick_vals, positions=[ia], widths=0.5, patch_artist=True, boxprops=dict(facecolor=area_color), showfliers=False, whis=[5, 95])

In [ ]:
# mixed encoding in SCRMN stronger than in VIS (looking at lick dropout scores for sensory cluster neurons)
fig, ax = plt.subplots()
area_drops = []
for ia, area in enumerate(['VISall', 'SCMRN']):
    unit_ids = vbn.get_unit_ids(units, areas=area, clusters='sensory', clustering='new')
    drops = dropouts_areas_edited[dropouts_areas_edited['unit_id'].isin(unit_ids)]["('absolute_change_from_full', 'licks')"].values
    area_drops.append(drops)
    ax.boxplot(-drops, positions=[ia], widths=0.5, patch_artist=True, boxprops=dict(facecolor='gray'), 
               showfliers=False, whis=[10, 90], medianprops=dict(color='k', linewidth=1), notch=True)

pval = scipy.stats.ranksums(area_drops[0], area_drops[1])
fig.suptitle(f'Lick dropout for sensory cluster; pval: {pval.pvalue:.2e}')

print(pval)
ax.set_xticks(np.arange(len(['VISall', 'SCm/MRN'])))
ax.set_xticklabels(['VISall', 'SCm/MRN'])
ax.set_ylabel('Lick dropout score')
vbn.formatFigure(fig, ax,) 

In [ ]:
import vbn_utils
for region in ['VISall', 'SCMRN']:

    good_units = vbn_utils.get_unit_ids(dropouts, region)
    region_dropouts = dropouts.set_index('unit_id').loc[good_units]

    # print(region, region_dropouts["('absolute_change_from_full', 'licks')"].mean())
    print(region, 'images', np.sum(region_dropouts["('absolute_change_from_full', 'all-images')"].values<-0.01)/len(region_dropouts), len(region_dropouts))
    print(region, 'licks', np.sum(region_dropouts["('absolute_change_from_full', 'licks')"].values<-0.01)/len(region_dropouts))